#Creating the DataWarehouse (SQL)

In [25]:
import sqlite3
db = sqlite3.connect("restaurant_warehouse.db")
db.executescript("""
CREATE TABLE IF NOT EXISTS Dim_Date (
    date_key INTEGER PRIMARY KEY,
    full_date DATE,
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER
);
CREATE TABLE IF NOT EXISTS Dim_Customer (
    customer_key INTEGER PRIMARY KEY,
    customer_id INTEGER,
    customer_name TEXT
);
CREATE TABLE IF NOT EXISTS Dim_Menu_Item (
    menu_key INTEGER PRIMARY KEY,
    item_id INTEGER,
    item_name TEXT,
    category TEXT
);
CREATE TABLE IF NOT EXISTS Fact_Restaurant_Sales (
    sales_key INTEGER PRIMARY KEY,
    date_key INTEGER,
    customer_key INTEGER,
    menu_key INTEGER,
    order_id INTEGER,
    sales_amount REAL,
    quantity INTEGER,
    FOREIGN KEY (date_key) REFERENCES Dim_Date(date_key),
    FOREIGN KEY (customer_key) REFERENCES Dim_Customer(customer_key),
    FOREIGN KEY (menu_key) REFERENCES Dim_Menu_Item(menu_key)
);
""")
db.commit()
print("All datawarehouse tables created successfully!")

All datawarehouse tables created successfully!


#Load and prepare the restaurant data

In [26]:
import pandas as pd
# Load datasets
orders = pd.read_csv("restaurant-sample-database-v1-csv-orders.csv")
items = pd.read_csv("restaurant-sample-database-v1-csv-order_items.csv")
customers = pd.read_csv("restaurant-sample-database-v1-csv-customers.csv")
menu = pd.read_csv("restaurant-sample-database-v1-csv-food_and_beverage_catalog.csv")
categories = pd.read_csv("restaurant-sample-database-v1-csv-food_category.csv")
payments = pd.read_csv("restaurant-sample-database-v1-csv-payments.csv")
# Display column names to verify the dataset
print("Orders:", orders.columns.tolist())
print("Items:", items.columns.tolist())
print("Customers:", customers.columns.tolist())
print("Menu:", menu.columns.tolist())
print("Categories:", categories.columns.tolist())
print("Payments:", payments.columns.tolist())

Orders: ['order_id', 'customer_id', 'delivery_boy_id', 'order_date', 'order_status', 'total_amount']
Items: ['order_item_id', 'item_id', 'order_id', 'price', 'quantity']
Customers: ['customer_id', 'address', 'email', 'first_name', 'last_name', 'phone', 'premium_status']
Menu: ['item_id', 'category_id', 'description', 'is_available', 'name', 'price', 'subcategory_id']
Categories: ['category_id', 'category_name']
Payments: ['payment_id', 'amount', 'order_id', 'payment_date', 'payment_method']


#Preprocessing + Data Warehouse

In [35]:

# 1. Merge datasets
df = items.merge(orders, on="order_id")
df = df.merge(customers, on="customer_id")
df = df.merge(menu, on="item_id")
df = df.merge(categories, on="category_id")
df = df.rename(columns={"price_x": "price"}).drop(
    columns=["price_y"], errors="ignore"
)

# 2. Preprocessing
df = df.drop_duplicates(subset=["order_item_id"])
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=[
    "order_id", "customer_id", "item_id",
    "order_date", "quantity", "price"
])
df = df[(df["quantity"] > 0) & (df["price"] >= 0)]
df["sales_amount"] = df["quantity"] * df["price"]

print("Preprocessing completed!")
print("Cleaned rows:", len(df))

# 3. Create Date Dimension
dim_date = pd.DataFrame({
    "full_date": df["order_date"].dt.normalize().drop_duplicates()
}).sort_values("full_date").reset_index(drop=True)

dim_date["date_key"] = (
    dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
)
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["year"] = dim_date["full_date"].dt.year

# 4. Create Customer Dimension
dim_customer = customers[
    ["customer_id", "first_name", "last_name"]
].drop_duplicates(subset=["customer_id"]).reset_index(drop=True)

dim_customer["customer_key"] = dim_customer.index + 1
dim_customer["customer_name"] = (
    dim_customer["first_name"].fillna("") + " " +
    dim_customer["last_name"].fillna("")
).str.strip()

# 5. Create Menu Dimension
dim_menu = menu[
    ["item_id", "name", "category_id"]
].merge(categories, on="category_id", how="left")

dim_menu = dim_menu.drop_duplicates(
    subset=["item_id"]
).reset_index(drop=True)

dim_menu["menu_key"] = dim_menu.index + 1
dim_menu = dim_menu.rename(columns={
    "name": "item_name",
    "category_name": "category"
})

# 6. Create Payment Dimension
dim_payment = payments[
    ["order_id", "payment_method"]
].dropna(subset=["payment_method"]).copy()

order_payment = dim_payment.drop_duplicates(
    subset=["order_id"]
).copy()

dim_payment = pd.DataFrame({
    "payment_method": order_payment["payment_method"].drop_duplicates()
}).reset_index(drop=True)

dim_payment["payment_key"] = dim_payment.index + 1

dim_payment = dim_payment[
    ["payment_key", "payment_method"]
]

# 7. Add dimension keys to fact data
fact = df.copy()
fact["full_date"] = fact["order_date"].dt.normalize()

fact = fact.merge(
    dim_date[["full_date", "date_key"]], on="full_date", how="left"
)
fact = fact.merge(
    dim_customer[["customer_id", "customer_key"]],
    on="customer_id", how="left"
)
fact = fact.merge(
    dim_menu[["item_id", "menu_key"]],
    on="item_id", how="left"
)
fact = fact.merge(
    order_payment[["order_id", "payment_method"]],
    on="order_id", how="left"
)
fact = fact.merge(
    dim_payment, on="payment_method", how="left"
)

fact["sales_key"] = range(1, len(fact) + 1)

# 8. Create and populate Data Warehouse
db = sqlite3.connect("restaurant_warehouse.db")

dim_date_to_save = dim_date.copy()
dim_date_to_save["full_date"] = (
    dim_date_to_save["full_date"].dt.strftime("%Y-%m-%d")
)

dim_date_to_save.to_sql(
    "Dim_Date", db, if_exists="replace", index=False
)

dim_customer[
    ["customer_key", "customer_id", "customer_name"]
].to_sql(
    "Dim_Customer", db, if_exists="replace", index=False
)

dim_menu[
    ["menu_key", "item_id", "item_name", "category"]
].to_sql(
    "Dim_Menu_Item", db, if_exists="replace", index=False
)

dim_payment.to_sql(
    "Dim_Payment", db, if_exists="replace", index=False
)

fact[
    ["sales_key", "date_key", "customer_key", "menu_key",
     "payment_key", "order_id", "sales_amount", "quantity"]
].to_sql(
    "Fact_Restaurant_Sales", db, if_exists="replace", index=False
)
print("Data Warehouse created and populated successfully!")


Preprocessing completed!
Cleaned rows: 1000
Data Warehouse created and populated successfully!


#Verify the warehouse

In [36]:
for table in [
    "Dim_Date",
    "Dim_Customer",
    "Dim_Menu_Item",
    "Dim_Payment",
    "Fact_Restaurant_Sales"
]:
    print("\n", table)
    print(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", db))


 Dim_Date
    full_date  date_key  day  month  quarter  year
0  2021-01-11  20210111   11      1        1  2021
1  2021-01-13  20210113   13      1        1  2021
2  2021-02-19  20210219   19      2        1  2021
3  2021-02-28  20210228   28      2        1  2021
4  2021-03-04  20210304    4      3        1  2021

 Dim_Customer
   customer_key  customer_id    customer_name
0             1            1    Orval Pacocha
1             2            2  Arnulfo Simonis
2             3            3     Lucie Beahan
3             4            4     Justyn Lakin
4             5            5       Teri Braun

 Dim_Menu_Item
   menu_key  item_id          item_name         category
0         1        1  Tomato Basil Soup  Pizza and Pasta
1         2        2  French Onion Soup  Pizza and Pasta
2         3        3         Minestrone  Pizza and Pasta
3         4        4       Caesar Salad  Pizza and Pasta
4         5        5        Greek Salad  Pizza and Pasta

 Dim_Payment
   payment_key payme

#OLAP Operations

In [37]:
orders.to_sql("orders", db, if_exists="replace", index=False)
payments.to_sql("payments", db, if_exists="replace", index=False)

# 1. ROLL-UP: Total sales by year
q1 = """
SELECT d.year, SUM(f.sales_amount) AS total_sales
FROM Fact_Restaurant_Sales f
JOIN Dim_Date d ON f.date_key = d.date_key
GROUP BY d.year
ORDER BY d.year
"""
print("ROLL-UP: Sales by Year")
print(pd.read_sql(q1, db))

# 2. DRILL-DOWN: Total sales by month
q2 = """
SELECT d.year, d.month, SUM(f.sales_amount) AS total_sales
FROM Fact_Restaurant_Sales f
JOIN Dim_Date d ON f.date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month
"""
print("\nDRILL-DOWN: Sales by Month")
print(pd.read_sql(q2, db))

# 3. SLICE: Sales by category for one year
q3 = """
SELECT m.category, SUM(f.sales_amount) AS total_sales
FROM Fact_Restaurant_Sales f
JOIN Dim_Date d ON f.date_key = d.date_key
JOIN Dim_Menu_Item m ON f.menu_key = m.menu_key
WHERE d.year = (SELECT MIN(year) FROM Dim_Date)
GROUP BY m.category
ORDER BY m.category
"""
print("\nSLICE: Sales by Category for One Year")
print(pd.read_sql(q3, db))

# 4. DICE: Sales by payment method for January and February
# Assumes one payment method per order.
q4 = """
SELECT d.month, p.payment_method,
       SUM(f.sales_amount) AS total_sales
FROM Fact_Restaurant_Sales f
JOIN Dim_Date d ON f.date_key = d.date_key
JOIN orders o ON f.order_id = o.order_id
JOIN payments p ON o.order_id = p.order_id
WHERE d.month IN (1, 2)
GROUP BY d.month, p.payment_method
ORDER BY d.month, p.payment_method
"""
print("\nDICE: Sales by Payment Method for January and February")
print(pd.read_sql(q4, db))

# 5. PIVOT: Category versus Month
q5 = """
SELECT m.category, d.month,
       SUM(f.sales_amount) AS total_sales
FROM Fact_Restaurant_Sales f
JOIN Dim_Date d ON f.date_key = d.date_key
JOIN Dim_Menu_Item m ON f.menu_key = m.menu_key
GROUP BY m.category, d.month
ORDER BY m.category, d.month
"""
result = pd.read_sql(q5, db)
print("\nPIVOT: Category vs Month")
print(
    result.pivot_table(
        index="category",
        columns="month",
        values="total_sales",
        aggfunc="sum",
        fill_value=0
    )
)
# Clean up temporary tables
cursor = db.cursor()
cursor.execute("DROP TABLE IF EXISTS orders")
cursor.execute("DROP TABLE IF EXISTS payments")
db.close()

ROLL-UP: Sales by Year
   year  total_sales
0  2021      1401.90
1  2022      2158.45
2  2023      2687.20
3  2024      4055.20
4  2025      4738.55

DRILL-DOWN: Sales by Month
    year  month  total_sales
0   2021      1        99.50
1   2021      2        73.20
2   2021      3       219.00
3   2021      4       115.30
4   2021      5        52.00
5   2021      6        49.50
6   2021      7       144.35
7   2021      8       236.50
8   2021      9        20.30
9   2021     10        93.00
10  2021     11       203.05
11  2021     12        96.20
12  2022      1       195.10
13  2022      2       193.00
14  2022      3       142.45
15  2022      4       115.00
16  2022      5       189.40
17  2022      6        40.00
18  2022      7       195.25
19  2022      8       121.60
20  2022      9       312.65
21  2022     10       223.00
22  2022     11       259.15
23  2022     12       171.85
24  2023      1       264.55
25  2023      2       266.00
26  2023      3       221.00
27  2023   